In [6]:
import os
import time
import requests
import psycopg2
from dotenv import load_dotenv
from pathlib import Path
import sys
from sqlalchemy import create_engine

def connect_to_database():
    DATABASE_URL = (
        "postgresql://postgres:1234"
        "@localhost:5432/google_and_spotify"
    )

    engine = create_engine(
        DATABASE_URL
    )
    return engine


load_dotenv()

# ============================================================
# CONFIGURAÇÕES
# ============================================================

GOOGLE_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")

TABLE_NAME = "sua_tabela"


# ============================================================
# GOOGLE PLACES API
# ============================================================

def get_place_details(place_id):
    """
    Consulta o Google Places API usando o Place ID.
    Retorna o nome e o link oficial do Google Maps.
    """

    url = f"https://places.googleapis.com/v1/places/{place_id}"

    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_API_KEY,
        "X-Goog-FieldMask": "id,displayName,googleMapsUri",
    }

    response = requests.get(url, headers=headers, timeout=10)

    if response.status_code == 200:
        data = response.json()

        name = data.get("displayName", {}).get("text")
        maps_url = data.get("googleMapsUri")

        return name, maps_url

    elif response.status_code == 404:
        print(f"Place ID não encontrado: {place_id}")
        return None, None

    else:
        print(
            f"Erro Google API ({response.status_code}) "
            f"para {place_id}: {response.text}"
        )

        return None, None


# ============================================================
# BANCO DE DADOS
# ============================================================

def create_columns(cursor):

    cursor.execute(f"""
        ALTER TABLE {TABLE_NAME}
        ADD COLUMN IF NOT EXISTS google_maps_coords_url TEXT,
        ADD COLUMN IF NOT EXISTS google_maps_place_url TEXT,
        ADD COLUMN IF NOT EXISTS google_maps_name TEXT;
    """)


def update_row(cursor, place_id, latitude, longitude, name, place_url):

    # Link usando coordenadas
    coords_url = (
        f"https://www.google.com/maps/search/?api=1"
        f"&query={latitude},{longitude}"
    )

    # Link usando Place ID
    place_id_url = (
        f"https://www.google.com/maps/search/?api=1"
        f"&query=Google"
        f"&query_place_id={place_id}"
    )

    cursor.execute(f"""
        UPDATE {TABLE_NAME}
        SET
            google_maps_coords_url = %s,
            google_maps_place_url = %s,
            google_maps_name = %s
        WHERE place_id = %s
    """, (
        coords_url,
        place_id_url,
        name,
        place_id
    ))


# ============================================================
# PROCESSAMENTO
# ============================================================

def main():

    if not GOOGLE_API_KEY:
        raise ValueError(
            "GOOGLE_MAPS_API_KEY não encontrada no arquivo .env"
        )

    conn = connect_to_database()
    cursor = conn.cursor()

    print("Criando/verificando colunas...")

    create_columns(cursor)
    conn.commit()

    # Pega somente registros que ainda precisam ser processados
    cursor.execute(f"""
        SELECT
            place_id,
            latitude,
            longitude
        FROM {TABLE_NAME}
        WHERE place_id IS NOT NULL
          AND (
              google_maps_name IS NULL
              OR google_maps_place_url IS NULL
              OR google_maps_coords_url IS NULL
          );
    """)

    rows = cursor.fetchall()

    print(f"{len(rows)} lugares para processar.\n")

    for i, (place_id, latitude, longitude) in enumerate(rows, start=1):

        print(
            f"[{i}/{len(rows)}] "
            f"Consultando {place_id}..."
        )

        try:

            name, google_maps_url = get_place_details(place_id)

            # Mesmo que a API não retorne o nome,
            # podemos continuar criando os links.
            update_row(
                cursor,
                place_id,
                latitude,
                longitude,
                name,
                google_maps_url
            )

            conn.commit()

            print(
                f"    Nome: {name}"
            )

            print(
                f"    Maps: {google_maps_url}"
            )

        except Exception as e:

            print(
                f"    ERRO: {e}"
            )

            conn.rollback()

        # Pequeno intervalo entre requisições
        time.sleep(0.1)

    cursor.close()
    conn.close()

    print("\nProcessamento concluído!")


if __name__ == "__main__":
    main()

AttributeError: 'Engine' object has no attribute 'cursor'

In [5]:
print(get_place_details(place_id="ChIJb4LZaFBoGZURQkQID5stwn0"))

('Subway', 'https://maps.google.com/?cid=9061855544218240066&g_mp=CiVnb29nbGUubWFwcy5wbGFjZXMudjEuUGxhY2VzLkdldFBsYWNlEAIYBCAA')
